In [ ]:
import requests
import pandas as pd
import numpy as np

In [ ]:
# Set up the request
base = "https://dwr.state.co.us/Rest/GET/api/v2/surfacewater/surfacewatertsday/?"
params = {
    "format": "json",
    "dateFormat": "spaceSepToSeconds",
    "abbrev": "PLASPLCO",
    "min-measDate": "04-01-2022",
    "max-measDate": "08-24-2026",
    "pageSize": 50000,
    "pageIndex": 1
}

# Execute the request
response = requests.get(base, params=params)
data = response.json()

# Print url for debugging
print(f"Requesting URL: {response.url}")

# Process the ResultList
if "ResultList" in data:
    df_dwr = pd.DataFrame(data["ResultList"])
    
    # 1. Clean up the dates and convert the value column to numeric
    df_dwr['measDate'] = pd.to_datetime(df_dwr['measDate'])
    df_dwr['value'] = pd.to_numeric(df_dwr['value'], errors='coerce')
    
    # 2. Filter for 'Streamflow' and keep only the essential columns
    df_flow = df_dwr[df_dwr['measType'] == 'Streamflow'].copy()
    
    # 3. Set the DatetimeIndex
    df_flow.set_index('measDate', inplace=True)
    df_flow = df_flow[['value']].rename(columns={'value': 'Flow_CFS'})
    
    # 4. Convert -999 flags to NaN using .replace()
    # This targets ONLY the values, leaving the row structure completely safe
    df_flow['Flow_CFS'] = df_flow['Flow_CFS'].replace(-999, np.nan)
    
    # 5. Force a continuous daily timeline from min to max date
    # This automatically injects rows with NaN for any completely missing days
    df_flow = df_flow.asfreq('D')
    
    # 6. Impute all NaNs (both missing days and the old -999s) to 0
    # print(f"Total missing or flagged days found: {df_flow['Flow_CFS'].isnull().sum()}")
    # df_flow['Flow_CFS'] = df_flow['Flow_CFS'].fillna(0)
    
    print("Success! Flow data loaded and regularized.")
    print(df_flow.tail())
else:
    print("Error: Could not find ResultList in the response.")

In [ ]:
df_flow.to_csv("fresh-data/SouthPlatteFlow.csv")

In [ ]:
# Set up the request to telemetry - includes preliminary weather data
base = "https://dwr.state.co.us/Rest/GET/api/v2/telemetrystations/telemetrytimeseriesday/?"
params = {
    "format": "json",
    "dateFormat": "spaceSepToSeconds",
    "abbrev": "PLASPLCO",
    "startDate": "04-01-2022",
    "endDate": "08-24-2026",
    "pageSize": 50000,
    "pageIndex": 1
}

# Execute the request
response = requests.get(base, params=params)
data = response.json()

# Print url for debugging
print(f"Requesting URL: {response.url}")

# Process the ResultList
if "ResultList" in data:

    df_dwr = pd.DataFrame(data["ResultList"])

    # Clean columns
    df_dwr['measDate'] = pd.to_datetime(df_dwr['measDate'])
    df_dwr['measValue'] = pd.to_numeric(df_dwr['measValue'], errors='coerce')

    # Keep desired measurements
    keep_types = ['DISCHRG', 'PRECIP', 'GAGE_HT']

    df_dwr = (
        df_dwr[df_dwr['parameter'].isin(keep_types)]
        .copy()
    )

    # Replace DWR missing value flag
    df_dwr['measValue'] = df_dwr['measValue'].replace(-999, np.nan)

    # Pivot long → wide
    df_flow = (
        df_dwr
        .pivot_table(
            index='measDate',
            columns='parameter',
            values='measValue',
            aggfunc='mean'
        )
        .sort_index()
    )

    # Rename columns
    df_flow = df_flow.rename(columns={
        'DISCHRG': 'Flow_CFS',
        'GAGE_HT': 'GageHeight_ft',
        'PRECIP': 'Precip'
    })

    # Create continuous daily index
    full_index = pd.date_range(
        start=df_flow.index.min(),
        end=df_flow.index.max(),
        freq='D'
    )

    df_flow = df_flow.reindex(full_index)
    df_flow.index.name = 'Date'
    
    print("Success! Telemetry data loaded and regularized.")
    print(df_flow.tail())
else:
    print("Error: Could not find ResultList in the response.")

In [ ]:
df_flow.to_csv("fresh-data/SouthPlatteTelemetry.csv")